In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prod"], label="select Environment")
env = dbutils.widgets.get("environment")
print(f"Environment: {env}")


In [0]:
tablName    = f"saleslake_{env}.bronze_{env}.rawProduct"
srcFileLoc  = f"s3://saleslake-101-251050869899-eu-north-1-an/saleslake-101/{env}/source_files/product/"
#s3://saleslake-101-251050869899-eu-north-1-an/saleslake-101/dev/source_files/product/
print(f"Target table : {tablName}")
print(f"Source files : {srcFileLoc}")

In [0]:
spark.sql(f"""
COPY INTO {tablName}
FROM (
    SELECT product_id, sku, product_name, category, sub_category, brand, supplier, unit_cost, list_price, launch_date, status,
           current_timestamp() AS ingest_ts
    FROM "{srcFileLoc}"
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS  ('mergeSchema' = 'false')
""")

print(f"Bronze load complete for {tablName}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {tablName}").display()


In [0]:
%sql
-- drop table saleslake_dev.bronze_dev.rawProduct;